# CP3/CP6 — Dashboard Streamlit

A identificação completa do grupo está na versão de entrega dos notebooks.

O CP3 reúne os indicadores do CP1 e os modelos do CP2 em uma aplicação Streamlit. O enunciado original está no material da disciplina; a cópia recuperada foi preservada junto aos documentos de revisão.

- [Abrir a aplicação](https://fiap-dsa-2ccph-2026-cerveja.streamlit.app/)
- [Ver o repositório](https://github.com/idrenanbr/fiap-dsa-2ccph-2026-cerveja)

O acesso sem login foi testado em 27/10/2026. Os relatórios de execução registram os testes locais, a execução em Linux e as verificações da aplicação hospedada.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT/'analysis.py').exists():
    raise RuntimeError('Execute o notebook a partir da pasta pacote_revisao')
sys.path.insert(0, str(ROOT))
from analysis import *
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
df, quality = load_data()
display(pd.Series(quality, name='Qualidade'))
display(df.head())

linhas_originais                                                         941
linhas_vazias_removidas                                                  576
linhas_validas                                                           365
ausentes                                                                   0
duplicadas                                                                 0
sha256_csv                 57a1df4479336ac4a6510fc7cc84f3a37eff88d0bdb915...
Name: Qualidade, dtype: object

,Data,Temperatura Media (C),Temperatura Minima (C),Temperatura Maxima (C),Precipitacao (mm),Final de Semana,Consumo de cerveja (litros)
0,2015-01-01,27.30,23.9,32.5,0.0,0,25.461
1,2015-01-02,27.02,24.5,33.5,0.0,0,28.972
2,2015-01-03,24.82,22.4,29.9,0.0,1,30.814
3,2015-01-04,23.98,21.5,28.6,1.2,1,29.799
4,2015-01-05,23.82,21.0,28.3,0.0,0,28.900


In [2]:
metrics=pd.read_csv(ROOT/'artifacts/metrics.csv'); display(metrics)
metadata=json.loads((ROOT/'artifacts/metadata.json').read_text(encoding='utf-8')); display(metadata)

,modelo,fase,MAE,RMSE,R2
0,Ridge,validacao,2.024107,2.405374,0.744962
1,XGBoost,validacao,2.010163,2.439080,0.737765
2,Rede profunda,validacao,2.060505,2.479096,0.729090
3,Ridge,teste,2.214502,2.618879,0.513616
4,XGBoost,teste,2.239193,2.610434,0.516748
5,Rede profunda,teste,2.209101,2.657089,0.499320
6,Baseline média,teste,3.448739,4.101331,-0.192883


{'seed': 42,
 'winner_validation_rmse': 'Ridge',
 'features': ['Temperatura Media (C)',
  'Temperatura Minima (C)',
  'Temperatura Maxima (C)',
  'Precipitacao (mm)',
  'Final de Semana'],
 'target': 'Consumo de cerveja (litros)',
 'split': {'treino': {'n': 219, 'inicio': '2015-01-01', 'fim': '2015-08-07'},
  'validacao': {'n': 73, 'inicio': '2015-08-08', 'fim': '2015-10-19'},
  'teste': {'n': 73, 'inicio': '2015-10-20', 'fim': '2015-12-31'}},
 'quality': {'linhas_originais': 941,
  'linhas_vazias_removidas': 576,
  'linhas_validas': 365,
  'ausentes': 0,
  'duplicadas': 0,
  'sha256_csv': '57a1df4479336ac4a6510fc7cc84f3a37eff88d0bdb915b2f122170401ab3a82'},
 'warnings': [],
 'trained_until': '2015-10-19',
 'note': 'Artefatos treinados em treino+validação; teste não usado no ajuste ou escolha.'}

## Código da aplicação

In [3]:
display(Markdown('```python\n'+(ROOT/'app.py').read_text(encoding='utf-8')+'\n```'))

```python
from pathlib import Path
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import streamlit as st
from analysis import ROOT, TARGET, FEATURES, SOURCE, load_data, indicators

st.set_page_config(page_title='Consumo de cerveja | São Paulo',layout='wide')
st.title('Consumo de cerveja em São Paulo')
st.caption('Data Science & Analytics · FIAP 2CCPH-2026 · CP3/CP6')
st.markdown('Dados diários de 2015, disponíveis no [Kaggle]('+SOURCE+'). Fonte consultada em 27/10/2026.')
st.info('Explore como o consumo variou com o clima e os dias da semana. Os resultados se referem a esta base de 2015; não representam toda a população de São Paulo.')

@st.cache_data
def data(): return load_data()
@st.cache_resource
def model(name): return joblib.load(ROOT/'artifacts'/(name.replace(' ','_')+'.joblib'))

df,quality=data()
meta=json.loads((ROOT/'artifacts/metadata.json').read_text(encoding='utf-8'))
metrics=pd.read_csv(ROOT/'artifacts/metrics.csv')
st.sidebar.header('Explorar dados')
period=st.sidebar.date_input('Período',value=(df.Data.min().date(),df.Data.max().date()),
                            min_value=df.Data.min().date(),max_value=df.Data.max().date())
day=st.sidebar.selectbox('Tipo de dia',['Todos','Dia útil','Final de semana'])
rain=st.sidebar.selectbox('Chuva',['Todos','Com chuva','Sem chuva'])
filtered=df.copy()
if len(period)==2:
    filtered=filtered[filtered.Data.between(pd.Timestamp(period[0]),pd.Timestamp(period[1]))]
else:
    st.sidebar.caption('Selecione também a data final.')
if day!='Todos': filtered=filtered[filtered['Final de Semana'].eq(int(day=='Final de semana'))]
if rain!='Todos': filtered=filtered[filtered['Precipitacao (mm)'].gt(0).eq(rain=='Com chuva')]
explore,predict,methods=st.tabs(['Dados e indicadores','Previsões','Sobre os dados'])
with explore:
    if filtered.empty:
        st.warning('Não há dados nesse recorte. Tente ampliar o período ou mudar os filtros.')
    else:
        k=indicators(filtered); cols=st.columns(3)
        cols[0].metric('Dias observados',k['dias'])
        cols[1].metric('Consumo médio · unidade do CSV',f"{k['consumo_medio']:.3f}")
        cols[2].metric('Variação no fim de semana (%)',f"{k['diferenca_fim_semana_pct']:.1f}" if np.isfinite(k['diferenca_fim_semana_pct']) else 'Não disponível')
        st.caption('A variação compara o consumo médio dos fins de semana com o dos dias úteis. Para calculá-la, o recorte precisa conter os dois tipos de dia.')
        st.subheader('Consumo ao longo do período')
        st.line_chart(filtered.set_index('Data')[[TARGET]],x_label='Data',y_label='Unidade do CSV')
        st.subheader('Temperatura máxima e consumo')
        fig,ax=plt.subplots(figsize=(8,3)); ax.scatter(filtered[FEATURES[2]],filtered[TARGET],alpha=.6)
        ax.set(xlabel='Temperatura máxima (°C)',ylabel='Consumo (unidade do CSV)'); st.pyplot(fig); plt.close(fig)
        st.subheader('Outros indicadores do recorte')
        st.dataframe(pd.DataFrame({'indicador':list(k),'valor':list(k.values())}),hide_index=True)
        st.dataframe(filtered,hide_index=True)
        st.download_button('Baixar dados filtrados',filtered.to_csv(index=False).encode('utf-8'),'dados_filtrados.csv','text/csv')
with predict:
    st.subheader('Como os modelos se saíram')
    st.dataframe(metrics,hide_index=True)
    st.caption('MAE e RMSE mostram o tamanho dos erros: quanto menores, melhor. No R², valores maiores indicam melhor desempenho. Esta comparação usa sempre o mesmo período de teste, independentemente dos filtros da aba de dados.')
    st.write('Modelo escolhido na validação:',meta['winner_validation_rmse'])
    st.dataframe(pd.read_csv(ROOT/'artifacts/predictions_test.csv'),hide_index=True)
    st.subheader('Estime o consumo de um dia')
    chosen=st.selectbox('Modelo',['Ridge','XGBoost','Rede profunda'],index=['Ridge','XGBoost','Rede profunda'].index(meta['winner_validation_rmse']))
    with st.form('scenario'):
        inputs={}
        for col in FEATURES[:-1]:
            inputs[col]=st.number_input(col,value=float(df[col].median()),min_value=0. if col=='Precipitacao (mm)' else -10.,max_value=300. if col=='Precipitacao (mm)' else 55.)
        inputs['Final de Semana']=int(st.checkbox('Final de semana'))
        submitted=st.form_submit_button('Estimar consumo')
    if submitted:
        if not inputs[FEATURES[1]]<=inputs[FEATURES[0]]<=inputs[FEATURES[2]]:
            st.error('A temperatura mínima deve ser menor ou igual à média, e a média menor ou igual à máxima.')
        else:
            row=pd.DataFrame([inputs],columns=FEATURES)
            outside=[c for c in FEATURES if inputs[c]<df[c].min() or inputs[c]>df[c].max()]
            if outside: st.warning('Os valores a seguir estão fora da faixa observada na base: '+', '.join(outside)+'. A previsão fica menos confiável nesse caso.')
            p=float(model(chosen).predict(row)[0])
            st.metric('Consumo estimado · unidade do CSV',f'{p:.3f}')
            if p<0: st.warning('O modelo retornou um valor negativo, que não faz sentido para consumo. Não use este resultado para planejar estoque.')
            st.caption('Este é um valor estimado, sem intervalo de previsão. Para planejar dias futuros, também seria preciso considerar o erro da previsão do tempo.')
with methods:
    st.write('Os números mantêm a escala do arquivo original, cuja coluna está rotulada como litros. A escala física precisa ser confirmada com a fonte antes de usar os resultados na operação de um estabelecimento.')
    st.write('O objetivo é entender a variação do consumo e avaliar se clima e calendário ajudam a estimar a demanda diária.')
    with st.expander('Qualidade dos dados e detalhes do treinamento'):
        st.json(quality)
        st.json(meta)
    st.markdown('Foram comparados três modelos: Ridge, XGBoost e uma rede neural com três camadas ocultas de 64, 32 e 16 neurônios. A escolha do modelo padrão foi feita na validação, antes de avaliar o teste.')
    st.markdown('A base cobre apenas um ano. Ela não informa a idade ou o perfil dos consumidores, os horários de consumo nem quais dias eram feriados. As relações encontradas são associações, não demonstrações de causa e efeito.')
    st.caption('Os nomes e RMs do grupo estão nos notebooks entregues para revisão.')

```

## Teste da interface

O teste abaixo usa AppTest para abrir a aplicação, mudar os filtros e preencher o formulário de previsão. Ele verifica o funcionamento da interface. O acesso pela Internet foi conferido separadamente depois da publicação, sem credenciais.

In [4]:
import subprocess
r=subprocess.run([sys.executable,str(ROOT/'test_app.py')],cwd=ROOT,text=True,capture_output=True)
print(r.stdout)
if r.returncode: print(r.stderr)
assert r.returncode==0, 'Teste da aplicação falhou'

{
  "status": "PASS",
  "testes": [
    "365 dias vÃ¡lidos",
    "3 artefatos carregam e predizem",
    "interface inicia sem exceÃ§Ãµes",
    "filtro fim de semana: 104 dias",
    "filtro chuva",
    "filtro vazio",
    "data Ãºnica",
    "previsÃ£o com cada modelo",
    "temperaturas inconsistentes rejeitadas"
  ]
}



## Como executar e acessar

As instruções locais estão no README.md. Com as dependências instaladas, execute `python -m streamlit run app.py`.

A aplicação está em https://fiap-dsa-2ccph-2026-cerveja.streamlit.app/ e o código em https://github.com/idrenanbr/fiap-dsa-2ccph-2026-cerveja. Os testes automatizados incluem a interface e a execução dos três notebooks. Os relatórios também registram o teste sem login e as previsões verificadas na hospedagem.

A revisão ainda não foi enviada ao Teams ou ao professor.